# 02 - Phase 1 Screening (4 models x 16 covariates x 2 windows x 2 horizons = 256 experiments)

Tujuan: screening single-covariate untuk mencari covariate mana yang memberi improvement signifikan terhadap baseline (target-only) di 4 model tree-based.

## Desain
- **Models (4):** RandomForest, ExtraTrees, XGBoost, LightGBM
- **Covariates (16):** None (baseline) + 15 single covariates
- **Windows (2):** 30, 120
- **Horizons (2):** 1 hari (next-day), 30 hari (one-month ahead)
- **CV:** 5-fold expanding window
- **Hyperparameters:** default konservatif dari spec (bukan arbitrary)
- **Metric utama:** MAPE di level harga IHSG (bukan log-diff)

## Output
- `phase1_screening_results.csv` - 256 rows, semua metrik per experiment
- `phase1_screening_analysis.csv` - pass/fail per covariate (threshold 0.3% improvement)

## 1. Imports & Load Data

In [ ]:
import pandas as pd
import numpy as np
import joblib
import gc
import glob
import traceback
from datetime import datetime
import warnings

from darts import TimeSeries
from darts.models import RandomForestModel, XGBModel, LightGBMModel
from darts.dataprocessing.transformers import Scaler, Diff
from darts.utils.missing_values import fill_missing_values
from sklearn.ensemble import ExtraTreesRegressor

try:
    from darts.models import SKLearnModel
except ImportError:
    from darts.models import RegressionModel as SKLearnModel

warnings.filterwarnings("ignore")

# Auto-load df_merged terbaru (tidak hardcode nama file)
joblib_files = sorted(glob.glob("saved_models/df_merged_*.joblib"), reverse=True)
if not joblib_files:
    raise FileNotFoundError("df_merged_*.joblib tidak ditemukan. Jalankan notebook 00 dulu.")
df_merged = joblib.load(joblib_files[0])
print(f"Loaded: {joblib_files[0]}")

LEVEL_VARS = ["M2", "USDIDR", "Coal", "Copper", "Nickel", "Silver", "Tin", "STI", "Gold", "WTI", "GDP"]
RATE_VARS  = ["BI_Rate", "CPI", "NPL_Ratio", "US_Treasury_10Y"]

print(f"Data: {df_merged.shape} | {df_merged['date'].min().date()} to {df_merged['date'].max().date()}")

## 2. Konfigurasi Eksperimen

In [ ]:
SINGLE_COVARIATES = {
    "None":            [],
    # Macro (7)
    "BI_Rate":         ["BI_Rate"],
    "CPI":             ["CPI"],
    "M2":              ["M2"],
    "NPL_Ratio":       ["NPL_Ratio"],
    "USDIDR":          ["USDIDR"],
    "GDP":             ["GDP"],
    "US_Treasury_10Y": ["US_Treasury_10Y"],
    # Commodity (7)
    "Coal":            ["Coal"],
    "Copper":          ["Copper"],
    "Nickel":          ["Nickel"],
    "Silver":          ["Silver"],
    "Tin":             ["Tin"],
    "Gold":            ["Gold"],
    "WTI":             ["WTI"],
    # Regional (1)
    "STI":             ["STI"],
}

WINDOWS  = [20, 120] # 1 bulan business day, 6 bulan business day
HORIZONS = [1, 20]     # 1 hari, 1 bulan business day
N_FOLDS  = 5

# Default hyperparameters (spec-based, bukan arbitrary)
MODEL_CONFIGS = {
    "RandomForest": {
        "n_estimators": 300,
        "max_depth": 5,
        "max_features": "sqrt",
        "max_samples": 0.7,
    },
    "ExtraTrees": {
        "n_estimators": 300,
        "max_depth": 5,
        "max_features": "sqrt",
    },
    "XGBoost": {
        "n_estimators": 300,
        "max_depth": 5,
        "learning_rate": 0.1,
        "subsample": 0.7,
        "colsample_bytree": 0.7,
        "reg_alpha": 0.01,
        "reg_lambda": 1.0,
    },
    "LightGBM": {
        "n_estimators": 300,
        "max_depth": 5,
        "learning_rate": 0.1,
        "num_leaves": 31,
        "subsample": 0.7,
        "colsample_bytree": 0.7,
        "reg_alpha": 0.01,
        "reg_lambda": 1.0,
    },
}

total_exp = len(MODEL_CONFIGS) * len(SINGLE_COVARIATES) * len(WINDOWS) * len(HORIZONS)
print(f"Models    : {list(MODEL_CONFIGS.keys())}")
print(f"Covariates: {len(SINGLE_COVARIATES)} configs")
print(f"Windows   : {WINDOWS}")
print(f"Horizons  : {HORIZONS}")
print(f"Total experiments: {total_exp}")

Models    : ['RandomForest', 'ExtraTrees', 'XGBoost', 'LightGBM']
Covariates: 16 configs
Windows   : [30, 180]
Horizons  : [1, 30]
Total experiments: 256


## 3. Helper Functions

In [7]:
def to_series(df, target_col, covariates=None):
    """Convert DataFrame to Darts TimeSeries (target + optional covariates)."""
    target = TimeSeries.from_dataframe(
        df, time_col="date", value_cols=target_col,
        fill_missing_dates=True, freq="B",
    )
    target = fill_missing_values(target)
    cov = None
    if covariates:
        cov = TimeSeries.from_dataframe(
            df, time_col="date", value_cols=covariates,
            fill_missing_dates=True, freq="B",
        )
        cov = fill_missing_values(cov)
    return target, cov


def build_model(model_name, window, horizon, has_covariates):
    """Factory: build a Darts model dengan window dan horizon yang ditentukan."""
    params = MODEL_CONFIGS[model_name]
    common = {
        "lags": window,
        "lags_past_covariates": window if has_covariates else None,
        "output_chunk_length": horizon,
    }
    if model_name == "RandomForest":
        return RandomForestModel(**common, random_state=42, n_jobs=-1, **params)
    elif model_name == "ExtraTrees":
        return SKLearnModel(
            **common,
            model=ExtraTreesRegressor(random_state=42, n_jobs=-1, **params),
        )
    elif model_name == "XGBoost":
        return XGBModel(**common, random_state=42, n_jobs=-1, **params)
    elif model_name == "LightGBM":
        return LightGBMModel(**common, random_state=42, n_jobs=-1, verbose=-1, **params)
    else:
        raise ValueError(f"Unknown model: {model_name}")


print("Helpers defined")

Helpers defined


## 4. Evaluate Function (5-fold Expanding CV, MAPE on Level)

Pipeline per fold:
1. Split expanding: train[:train_end], test[train_end:test_end]
2. Transform target: `log -> Diff(1) -> Scaler` (fit on train)
3. Transform covariates: `log -> Diff(1)` untuk LEVEL_VARS, `Diff(1)` untuk RATE_VARS, lalu `Scaler`
4. Fit model on train, historical forecast on test window
5. Inverse: scaler -> log-diff -> cumsum dari anchor -> exp -> harga level
6. Compute metrics: MAPE, MAE, RMSE, R2, DA


In [8]:
def train_and_evaluate(model_name, target_ts, cov_ts, window, horizon, n_folds=N_FOLDS):
    """5-fold expanding CV. Returns dict of per-fold metrics."""
    n = len(target_ts)
    test_size = int(n * 0.15)
    min_train = int(n * 0.4)
    available = n - min_train - test_size
    step = max(1, available // max(1, n_folds - 1))

    fold_metrics = {"mape": [], "mae": [], "rmse": [], "r2": [], "da": []}

    for fold in range(n_folds):
        train_end = min_train + fold * step
        test_end = min(train_end + test_size, n)
        if test_end > n or train_end >= test_end:
            break

        train_ts = target_ts[:train_end]
        test_ts  = target_ts[train_end:test_end]
        fold_ts  = target_ts[:test_end]

        # Target transform: log -> diff -> scale (fit on train)
        train_log = train_ts.map(np.log)
        fold_log  = fold_ts.map(np.log)
        differencer = Diff(lags=1)
        train_log_diff = differencer.fit_transform(train_log)
        fold_log_diff  = differencer.transform(fold_log)
        scaler = Scaler()
        train_scaled = scaler.fit_transform(train_log_diff)
        fold_scaled  = scaler.transform(fold_log_diff)

        # Covariate transform
        cov_transformed = None
        if cov_ts is not None:
            cov_fold  = cov_ts[:test_end]
            cov_train = cov_ts[:train_end]
            cov_cols   = cov_ts.components.tolist()
            level_cols = [c for c in cov_cols if c in LEVEL_VARS]
            rate_cols  = [c for c in cov_cols if c in RATE_VARS]

            parts_train, parts_fold = [], []
            if level_cols:
                d1 = Diff(lags=1)
                parts_train.append(d1.fit_transform(cov_train[level_cols].map(np.log)))
                parts_fold.append(d1.transform(cov_fold[level_cols].map(np.log)))
            if rate_cols:
                d2 = Diff(lags=1)
                parts_train.append(d2.fit_transform(cov_train[rate_cols]))
                parts_fold.append(d2.transform(cov_fold[rate_cols]))

            ct = parts_train[0]
            cf = parts_fold[0]
            for pt, pf in zip(parts_train[1:], parts_fold[1:]):
                ct = ct.stack(pt)
                cf = cf.stack(pf)

            cov_scaler = Scaler()
            cov_scaler.fit(ct)
            cov_transformed = cov_scaler.transform(cf)

        # Fit model (sekarang terima horizon)
        model = build_model(model_name, window, horizon, has_covariates=(cov_ts is not None))
        model.fit(train_scaled, past_covariates=cov_transformed)

        # Historical forecast — stride = horizon agar non-overlapping
        forecast_list = model.historical_forecasts(
            series=fold_scaled,
            past_covariates=cov_transformed,
            start=test_ts.start_time(),
            forecast_horizon=horizon,
            stride=horizon,
            retrain=False,
            last_points_only=False,
            verbose=False,
        )
        if isinstance(forecast_list, TimeSeries):
            forecast_list = [forecast_list]

        # Inverse transform ke level harga IHSG
        all_dates, all_prices = [], []
        fold_log_full = fold_ts.map(np.log)
        for chunk_scaled in forecast_list:
            chunk_diff = scaler.inverse_transform(chunk_scaled)
            dates = chunk_diff.time_index
            vals  = chunk_diff.values().flatten()
            idx   = fold_ts.get_index_at_point(dates[0])
            if idx == 0:
                continue
            anchor     = fold_log_full[idx - 1].values()[0][0]
            log_prices = anchor + np.cumsum(vals)
            all_dates.extend(dates)
            all_prices.extend(np.exp(log_prices))

        # Metrics
        pred_df   = pd.DataFrame({"date": pd.to_datetime(all_dates), "predicted": all_prices})
        actual_df = fold_ts.to_dataframe().reset_index()
        actual_df.columns = ["date", "actual"]
        eval_df = pd.merge(actual_df, pred_df, on="date", how="inner")
        y_true  = eval_df["actual"].values
        y_pred  = eval_df["predicted"].values

        mape   = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
        mae    = np.mean(np.abs(y_true - y_pred))
        rmse   = np.sqrt(np.mean((y_true - y_pred) ** 2))
        ss_res = np.sum((y_true - y_pred) ** 2)
        ss_tot = np.sum((y_true - y_true.mean()) ** 2)
        r2     = 1 - (ss_res / ss_tot) if ss_tot > 0 else np.nan
        actual_dir = np.diff(y_true)
        pred_dir   = y_pred[1:] - y_true[:-1]
        da = np.mean((actual_dir > 0) == (pred_dir > 0)) * 100 if len(actual_dir) > 0 else np.nan

        fold_metrics["mape"].append(mape)
        fold_metrics["mae"].append(mae)
        fold_metrics["rmse"].append(rmse)
        fold_metrics["r2"].append(r2)
        fold_metrics["da"].append(da)

        del model
        gc.collect()

    return fold_metrics


print("Evaluation function defined")

Evaluation function defined


## 5. Run Experiment Loop (128 experiments)

In [9]:
results = []
start_time = datetime.now()
exp_num = 0
print(f"Phase 1 Screening started: {start_time.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Total experiments: {total_exp}")
print("=" * 75)

for model_name in MODEL_CONFIGS.keys():
    for cov_name, cov_vars in SINGLE_COVARIATES.items():
        for window in WINDOWS:
            for horizon in HORIZONS:
                exp_num += 1
                tag = f"[{exp_num}/{total_exp}] {model_name:13s} | {cov_name:16s} | W{window}_H{horizon}"
                try:
                    target_ts, cov_ts = to_series(
                        df_merged, "IHSG", cov_vars if cov_vars else None
                    )
                    fm = train_and_evaluate(model_name, target_ts, cov_ts, window, horizon)

                    row = {
                        "Model":     model_name,
                        "Covariates": cov_name,
                        "Window":    window,
                        "Horizon":   horizon,
                        "MAPE_mean": round(np.mean(fm["mape"]), 4),
                        "MAPE_std":  round(np.std(fm["mape"]), 4),
                        "MAE_mean":  round(np.mean(fm["mae"]), 4),
                        "RMSE_mean": round(np.mean(fm["rmse"]), 4),
                        "RMSE_std":  round(np.std(fm["rmse"]), 4),
                        "R2_mean":   round(np.mean(fm["r2"]), 4),
                        "DA_mean":   round(np.mean(fm["da"]), 2),
                        "DA_std":    round(np.std(fm["da"]), 2),
                    }
                    results.append(row)
                    print(f"{tag} -> MAPE={row['MAPE_mean']:.4f}+-{row['MAPE_std']:.4f}% | DA={row['DA_mean']:.1f}%")
                except Exception as e:
                    print(f"{tag} -> ERROR: {e}")
                    traceback.print_exc()
                    results.append({
                        "Model": model_name, "Covariates": cov_name,
                        "Window": window, "Horizon": horizon, "Error": str(e),
                    })

elapsed = datetime.now() - start_time
print("=" * 75)
print(f"Done in {elapsed}")

df_results = pd.DataFrame(results)
df_results.to_csv("phase1_screening_results.csv", index=False)
print(f"Saved: phase1_screening_results.csv ({len(df_results)} rows)")
display(df_results.sort_values("MAPE_mean").head(20))

Phase 1 Screening started: 2026-05-04 12:45:12
Total experiments: 256
[1/256] RandomForest  | None             | W30_H1 -> MAPE=0.6666+-0.1721% | DA=50.7%
[2/256] RandomForest  | None             | W30_H30 -> MAPE=2.6953+-0.9643% | DA=49.6%
[3/256] RandomForest  | None             | W120_H1 -> MAPE=0.6626+-0.1674% | DA=52.3%
[4/256] RandomForest  | None             | W120_H30 -> MAPE=2.7055+-0.9644% | DA=50.0%
[5/256] RandomForest  | BI_Rate          | W30_H1 -> MAPE=0.6645+-0.1696% | DA=51.4%
[6/256] RandomForest  | BI_Rate          | W30_H30 -> MAPE=2.6875+-0.9543% | DA=49.7%
[7/256] RandomForest  | BI_Rate          | W120_H1 -> MAPE=0.6629+-0.1684% | DA=50.2%
[8/256] RandomForest  | BI_Rate          | W120_H30 -> MAPE=2.7040+-0.9645% | DA=49.9%
[9/256] RandomForest  | CPI              | W30_H1 -> MAPE=0.6636+-0.1691% | DA=52.3%
[10/256] RandomForest  | CPI              | W30_H30 -> MAPE=2.6900+-0.9548% | DA=49.7%
[11/256] RandomForest  | CPI              | W120_H1 -> MAPE=0.6615+-0.

,Model,Covariates,Window,Horizon,MAPE_mean,MAPE_std,MAE_mean,RMSE_mean,RMSE_std,R2_mean,DA_mean,DA_std
108,ExtraTrees,Silver,30,1,0.6581,0.1624,40.2529,55.0152,9.3042,0.9692,54.10,3.16
120,ExtraTrees,WTI,30,1,0.6583,0.1623,40.2698,55.0262,9.2846,0.9691,51.60,2.05
116,ExtraTrees,Gold,30,1,0.6583,0.1622,40.2697,55.0353,9.3001,0.9691,52.88,2.77
100,ExtraTrees,Copper,30,1,0.6586,0.1628,40.2804,55.0618,9.3728,0.9692,51.70,2.55
110,ExtraTrees,Silver,120,1,0.6586,0.1634,40.2849,55.0116,9.2925,0.9691,53.23,1.90
122,ExtraTrees,WTI,120,1,0.6586,0.1629,40.2885,55.0556,9.2997,0.9691,52.16,2.00
90,ExtraTrees,GDP,120,1,0.6587,0.1633,40.2862,55.0826,9.3521,0.9691,52.52,1.83
102,ExtraTrees,Copper,120,1,0.6590,0.1633,40.3079,55.0752,9.3508,0.9691,52.57,1.60
92,ExtraTrees,US_Treasury_10Y,30,1,0.6590,0.1632,40.3061,55.0489,9.2924,0.9690,52.26,2.36
74,ExtraTrees,CPI,120,1,0.6590,0.1629,40.3148,55.1087,9.3155,0.9690,51.76,2.17


## 6. Screening Analysis

Covariate dianggap "lolos" kalau improvement MAPE terhadap baseline (`None`) >= threshold. Default threshold = **0.3%** (relative improvement).


In [13]:
def analyze_screening(df, threshold_pct=0.3):
    rows = []
    for model_name in df["Model"].unique():
        for window in df["Window"].unique():
            for horizon in df["Horizon"].unique():
                mask = (
                    (df["Model"]   == model_name) &
                    (df["Window"]  == window) &
                    (df["Horizon"] == horizon)
                )
                subset = df[mask]
                baseline_row = subset[subset["Covariates"] == "None"]
                if baseline_row.empty or "MAPE_mean" not in baseline_row.columns:
                    continue
                baseline_mape = baseline_row["MAPE_mean"].values[0]

                for _, row in subset.iterrows():
                    if row["Covariates"] == "None":
                        continue
                    if pd.isna(row.get("MAPE_mean")):
                        continue
                    impr = (baseline_mape - row["MAPE_mean"]) / baseline_mape * 100
                    rows.append({
                        "Model":          model_name,
                        "Window":         window,
                        "Horizon":        horizon,
                        "Covariate":      row["Covariates"],
                        "MAPE":           row["MAPE_mean"],
                        "Baseline_MAPE":  baseline_mape,
                        "Improvement_pct": round(impr, 3),
                        "Passed":         impr >= threshold_pct,
                    })
    return pd.DataFrame(rows)


df_screening = analyze_screening(df_results, threshold_pct=0.3)
df_screening.to_csv("phase1_screening_analysis.csv", index=False)
print(f"Saved: phase1_screening_analysis.csv ({len(df_screening)} rows)")

# Pass counts (out of 4 models x 2 windows x 2 horizons = 16 combos)
max_combos = df_results["Model"].nunique() * df_results["Window"].nunique() * df_results["Horizon"].nunique()
pass_counts = (
    df_screening[df_screening["Passed"]]
    .groupby("Covariate").size()
    .sort_values(ascending=False)
    .rename("pass_count")
    .to_frame()
)
pass_counts["pass_rate"] = (pass_counts["pass_count"] / max_combos * 100).round(1)
print(f"\nCovariate pass counts (out of {max_combos} model x window x horizon combos):")
display(pass_counts)

# Ringkasan per horizon
print("\nPass counts breakdown per Horizon:")
for h in sorted(df_screening["Horizon"].unique()):
    sub = df_screening[df_screening["Horizon"] == h]
    pc  = sub[sub["Passed"]].groupby("Covariate").size().sort_values(ascending=False)
    max_h = df_results["Model"].nunique() * df_results["Window"].nunique()
    print(f"\n  H{h} (max {max_h} combos):")
    print(f"  {pc.to_dict()}")

Saved: phase1_screening_analysis.csv (240 rows)

Covariate pass counts (out of 16 model x window x horizon combos):


,pass_count,pass_rate
Covariate,,
USDIDR,8,50.0
Gold,7,43.8
Silver,7,43.8
WTI,7,43.8
BI_Rate,5,31.2
CPI,5,31.2
Copper,5,31.2
GDP,4,25.0
M2,4,25.0



Pass counts breakdown per Horizon:

  H1 (max 8 combos):
  {'Silver': 4, 'USDIDR': 4, 'Copper': 3, 'Gold': 3, 'Nickel': 3, 'STI': 3, 'BI_Rate': 2, 'GDP': 2, 'M2': 2, 'NPL_Ratio': 2, 'WTI': 2, 'CPI': 1, 'Tin': 1}

  H30 (max 8 combos):
  {'WTI': 5, 'CPI': 4, 'Gold': 4, 'USDIDR': 4, 'BI_Rate': 3, 'Silver': 3, 'US_Treasury_10Y': 3, 'Copper': 2, 'GDP': 2, 'M2': 2, 'NPL_Ratio': 2, 'Nickel': 1, 'STI': 1, 'Tin': 1}
